# perteval Quickstart Tutorial

> A unified evaluation framework for single-cell perturbation prediction

This notebook walks through the full perteval workflow:

1. **Create synthetic data** — generate a realistic AnnData dataset
2. **Split data** — train/val/test with transfer splitting
3. **Train a baseline model** — MeanControl (control cell mean)
4. **Evaluate predictions** — compute metrics per perturbation
5. **Export results** — JSON + CSV
6. **Run a full benchmark** — BenchmarkRunner orchestration
7. **Compare models** — multi-model comparison table

In [23]:
import perteval

print(f"perteval v{perteval.__version__}")

perteval v0.1.0


## 1. Create Synthetic Data

We generate a realistic single-cell dataset with:
- 1000 cells across 200 genes
- 5 perturbations + control
- 2 cell types
- Expression shifts per perturbation (simulating real perturbation effects)

In [24]:
import anndata as ad
import numpy as np

rng = np.random.default_rng(42)

n_obs, n_vars = 1000, 200
perturbations = ["IFNG", "TNF", "TGFB1", "IL6", "STAT1"]
cell_types = ["T_cell", "Monocyte"]

# Assign random labels
all_labels = ["control"] + perturbations
obs_labels = rng.choice(all_labels, size=n_obs)
obs_cell_types = rng.choice(cell_types, size=n_obs)

# Generate expression matrix with perturbation-specific shifts
X = rng.standard_normal((n_obs, n_vars)).astype(np.float32)
for i, pert in enumerate(perturbations, start=1):
    X[obs_labels == pert] += i * 0.5

adata = ad.AnnData(
    X=X,
    obs={"perturbation": obs_labels, "cell_type": obs_cell_types},
)
adata.var_names = [f"gene_{i}" for i in range(n_vars)]
adata.obs_names = [f"cell_{i}" for i in range(n_obs)]

print(adata)
print(f"\nPerturbation labels: {adata.obs['perturbation'].unique().tolist()}")
print(f"Cell types: {adata.obs['cell_type'].unique().tolist()}")
print(f"\nCells per condition:")
print(adata.obs["perturbation"].value_counts())

AnnData object with n_obs × n_vars = 1000 × 200
    obs: 'perturbation', 'cell_type'

Perturbation labels: ['control', 'IL6', 'TGFB1', 'TNF', 'STAT1', 'IFNG']
Cell types: ['Monocyte', 'T_cell']

Cells per condition:
perturbation
control    182
IL6        177
TGFB1      173
STAT1      159
TNF        157
IFNG       152
Name: count, dtype: int64


## 2. Split Data

perteval supports two splitting strategies:
- **`random`** — shuffle and split cells (i.i.d.)
- **`transfer`** — hold out entire perturbations (o.o.d., tests generalization to unseen perturbations)

We use `transfer` split here — the test set contains perturbations never seen during training.

In [25]:
from perteval.data import Splitter

splits = Splitter.split(
    adata,
    method="transfer",
    holdout_key="perturbation",
    frac=(0.6, 0.2, 0.2),
    seed=42,
)

for name, split_adata in splits.items():
    perts = [p for p in split_adata.obs["perturbation"].unique() if p != "control"]
    print(f"{name:>5}: {split_adata.n_obs} cells, perturbations: {perts}")

train: 650 cells, perturbations: ['TNF', 'STAT1', 'IFNG']
  val: 355 cells, perturbations: ['TGFB1']
 test: 359 cells, perturbations: ['IL6']


## 3. Train a Baseline Model

`MeanControl` is the simplest baseline — it predicts that every perturbed cell will look like the mean of control cells. This is the lower bound that any real model should beat.

In [26]:
from perteval.models.baselines.mean_control import MeanControl

model = MeanControl()
model.train(splits["train"], perturbation_key="perturbation", control_key="control")

# Predict on held-out perturbations using control cells from the test set
test_adata = splits["test"]
control_cells = test_adata[test_adata.obs["perturbation"] == "control"]
test_perts = [p for p in test_adata.obs["perturbation"].unique() if p != "control"]

predicted = model.predict(control_cells, perturbations=test_perts)

print(f"Predicted: {predicted}")
print(f"Test perturbations: {test_perts}")

Predicted: AnnData object with n_obs × n_vars = 1 × 200
    obs: 'perturbation'
Test perturbations: ['IL6']


## 4. Evaluate Predictions

Create a `PerturbationData` pair (validates gene alignment automatically), then run the `Evaluator` with selected metrics.

**Available metrics:**
- **Expression:** `pearson_delta`, `mse`, `mae` — compare mean expression vectors
- **DE:** `overlap_at_k` — overlap of top differentially expressed genes
- **Distribution:** `edistance` — energy distance between cell populations

In [27]:
from perteval import PerturbationData, Evaluator

# Build ground truth from test set (only perturbed cells)
gt_mask = test_adata.obs["perturbation"].isin(test_perts)
ground_truth = test_adata[gt_mask].copy()

# PerturbationData validates gene alignment and perturbation labels
data = PerturbationData(predicted=predicted, ground_truth=ground_truth)

print(f"Perturbations to evaluate: {data.perturbation_labels}")
print(f"Genes: {len(data.gene_names)}")

Perturbations to evaluate: ['IL6']
Genes: 200


In [28]:
evaluator = Evaluator()
result = evaluator.evaluate(
    data,
    metrics=["pearson_delta", "mse", "mae", "overlap_at_k", "edistance"],
)

print("=== Per-Perturbation Results ===")
print(result.per_perturbation)
print("\n=== Aggregated Statistics ===")
print(result.aggregated)

=== Per-Perturbation Results ===
shape: (1, 6)
┌──────────────┬───────────────┬──────────┬──────────┬──────────────┬───────────┐
│ perturbation ┆ pearson_delta ┆ mse      ┆ mae      ┆ overlap_at_k ┆ edistance │
│ ---          ┆ ---           ┆ ---      ┆ ---      ┆ ---          ┆ ---       │
│ str          ┆ f64           ┆ f64      ┆ f64      ┆ f64          ┆ f64       │
╞══════════════╪═══════════════╪══════════╪══════════╪══════════════╪═══════════╡
│ IL6          ┆ -0.0098       ┆ 3.967505 ┆ 1.989385 ┆ 0.55         ┆ 43.077676 │
└──────────────┴───────────────┴──────────┴──────────┴──────────────┴───────────┘

=== Aggregated Statistics ===
shape: (4, 6)
┌───────────┬───────────────┬──────────┬──────────┬──────────────┬───────────┐
│ statistic ┆ pearson_delta ┆ mse      ┆ mae      ┆ overlap_at_k ┆ edistance │
│ ---       ┆ ---           ┆ ---      ┆ ---      ┆ ---          ┆ ---       │
│ str       ┆ f64           ┆ f64      ┆ f64      ┆ f64          ┆ f64       │
╞═══════════╪═════

## 5. Export Results

Results can be saved as JSON (canonical, self-describing) or CSV (human-friendly).

In [29]:
import json
from pathlib import Path

output_dir = Path("_output")
output_dir.mkdir(exist_ok=True)

# JSON — includes config metadata for reproducibility
result.to_json(str(output_dir / "mean_control_result.json"))

# CSV — two files: per-perturbation + aggregated
result.to_csv(str(output_dir / "mean_control"))

# Peek at the JSON structure
with open(output_dir / "mean_control_result.json") as f:
    data_json = json.load(f)

print("JSON keys:", list(data_json.keys()))
print("\nConfig:")
for k, v in data_json["config"].items():
    print(f"  {k}: {v}")

JSON keys: ['config', 'per_perturbation', 'aggregated']

Config:
  metrics: ['pearson_delta', 'mse', 'mae', 'overlap_at_k', 'edistance']
  aggregation: average
  n_perturbations: 1
  n_genes: 200
  timestamp: 2026-04-08T02:48:52.495852+00:00


## 6. Run a Full Benchmark

`BenchmarkRunner` automates the entire pipeline: load data from YAML config, split, train, predict, evaluate. Just point it at a benchmarks directory and a data directory.

In [30]:
# First, save our synthetic data as h5ad so BenchmarkRunner can load it
data_dir = output_dir / "data"
data_dir.mkdir(exist_ok=True)
adata.write_h5ad(data_dir / "synthetic.h5ad")

# Create a benchmark YAML
bench_dir = output_dir / "benchmarks"
bench_dir.mkdir(exist_ok=True)
(bench_dir / "synthetic_bench.yaml").write_text("""
dataset: synthetic
metrics: [pearson_delta, mse, mae, edistance]
split:
  method: transfer
  holdout_key: perturbation
  frac: [0.6, 0.2, 0.2]
  seed: 42
aggregation: average
""".strip())

print("Benchmark YAML:")
print((bench_dir / "synthetic_bench.yaml").read_text())

Benchmark YAML:
dataset: synthetic
metrics: [pearson_delta, mse, mae, edistance]
split:
  method: transfer
  holdout_key: perturbation
  frac: [0.6, 0.2, 0.2]
  seed: 42
aggregation: average


In [31]:
from perteval.bench import BenchmarkRunner

runner = BenchmarkRunner(
    benchmarks=["synthetic_bench"],
    models=["mean_control"],
    benchmarks_dir=str(bench_dir),
    data_dir=str(data_dir),
)

results = runner.run()

# Access results
bench_result = results["synthetic_bench"]["mean_control"]
print("=== BenchmarkRunner Results ===")
print(bench_result.per_perturbation)
print(f"\nConfig: {bench_result.config}")

=== BenchmarkRunner Results ===
shape: (1, 5)
┌──────────────┬───────────────┬──────────┬──────────┬───────────┐
│ perturbation ┆ pearson_delta ┆ mse      ┆ mae      ┆ edistance │
│ ---          ┆ ---           ┆ ---      ┆ ---      ┆ ---       │
│ str          ┆ f64           ┆ f64      ┆ f64      ┆ f64       │
╞══════════════╪═══════════════╪══════════╪══════════╪═══════════╡
│ IL6          ┆ -0.0098       ┆ 3.967505 ┆ 1.989385 ┆ 43.077676 │
└──────────────┴───────────────┴──────────┴──────────┴───────────┘

Config: {'metrics': ['pearson_delta', 'mse', 'mae', 'edistance'], 'aggregation': 'average', 'n_perturbations': 1, 'n_genes': 200, 'timestamp': '2026-04-08T02:48:52.536152+00:00', 'benchmark': 'synthetic_bench', 'model': 'mean_control'}


## 7. Compare Models

`Compare` builds a summary table from the nested results dict. In a real scenario, you'd have multiple models to compare.

In [32]:
from perteval.bench import Compare

comparison = Compare.from_results(results)
print("=== Model Comparison ===")
print(comparison.summary())

=== Model Comparison ===
shape: (1, 6)
┌─────────────────┬──────────────┬───────────────┬──────────┬──────────┬───────────┐
│ benchmark       ┆ model        ┆ pearson_delta ┆ mse      ┆ mae      ┆ edistance │
│ ---             ┆ ---          ┆ ---           ┆ ---      ┆ ---      ┆ ---       │
│ str             ┆ str          ┆ f64           ┆ f64      ┆ f64      ┆ f64       │
╞═════════════════╪══════════════╪═══════════════╪══════════╪══════════╪═══════════╡
│ synthetic_bench ┆ mean_control ┆ -0.0098       ┆ 3.967505 ┆ 1.989385 ┆ 43.077676 │
└─────────────────┴──────────────┴───────────────┴──────────┴──────────┴───────────┘


## 8. Using Individual Metrics Directly

You can also use the functional metrics standalone — they're pure numpy/scipy functions with no framework overhead.

In [33]:
from perteval.metrics.functional.expression import pearson_delta, mse, mae
from perteval.metrics.functional.distribution import edistance
from perteval.metrics.functional.de import overlap_at_k

# Expression metrics — compare mean expression vectors
pred_mean = np.array([1.0, 2.5, 0.3, 4.1, 2.0])
true_mean = np.array([1.1, 2.4, 0.5, 3.9, 2.1])

print(f"Pearson delta: {pearson_delta(pred_mean, true_mean):.4f}")
print(f"MSE:           {mse(pred_mean, true_mean):.4f}")
print(f"MAE:           {mae(pred_mean, true_mean):.4f}")

# DE metric — compare ranked gene lists
pred_genes = np.array(["TP53", "BRCA1", "MYC", "EGFR", "KRAS"])
true_genes = np.array(["TP53", "MYC", "BRCA1", "PTEN", "KRAS"])
print(f"\nOverlap@3:     {overlap_at_k(pred_genes, true_genes, k=3):.4f}")
print(f"Overlap@5:     {overlap_at_k(pred_genes, true_genes, k=5):.4f}")

# Distribution metric — compare cell populations
rng = np.random.default_rng(42)
pop_a = rng.standard_normal((50, 10))
pop_b = rng.standard_normal((50, 10)) + 0.5
print(f"\nE-distance:    {edistance(pop_a, pop_b):.4f}")

Pearson delta: 0.9992
MSE:           0.0220
MAE:           0.1400

Overlap@3:     1.0000
Overlap@5:     0.8000

E-distance:    0.6045


## 9. Discover Available Resources

Use the registry to discover what's available.

In [34]:
from perteval.metrics import metric_registry
from perteval.models import model_registry

print("Available metrics:")
for name in metric_registry.list_available():
    info = metric_registry.get(name)
    print(f"  {name:20s} ({info.metric_type.value:12s}) best={info.best_value.value:4s}  {info.description}")

print(f"\nAvailable models:")
for name in model_registry.list_available():
    print(f"  {name}")

Available metrics:
  edistance            (distribution) best=zero  Energy distance between predicted and ground-truth cell distributions
  mae                  (expression  ) best=zero  Mean absolute error of mean expression
  mse                  (expression  ) best=zero  Mean squared error of mean expression
  overlap_at_k         (de          ) best=one   Fraction of top-k DE genes overlapping between predicted and ground-truth
  pearson_delta        (expression  ) best=one   Pearson correlation of mean expression shift

Available models:
  mean_control


## Cleanup

In [35]:
# import shutil

# shutil.rmtree(output_dir, ignore_errors=True)
# print("Cleaned up _output/")